# Exercise 12.1: From conductances to the action potential

In this exercise, we will progressively build up from a simple two-current model to the full Hodgkin-Huxley action potential. Each step adds a new layer of biophysical realism, and you will see exactly _why_ each layer is necessary to generate the characteristic spike.


## Exercise 12.1a: Fixed conductances cannot generate a spike

We begin with the simplest possible membrane model: two ionic currents with **fixed** (non-gated) conductances.

$$
C_{\mathrm{m}} \frac{\mathrm{d}V}{\mathrm{d}t} = -g_{\mathrm{Na}}(V - E_{\mathrm{Na}}) - g_{\mathrm{K}}(V - E_{\mathrm{K}}) + I_{\mathrm{app}}
$$

Use the widget below to explore this model. For each task, try to predict the answer analytically before using the widget to verify.

**Tasks:**

1. Set $g_{\mathrm{Na}} = g_{\mathrm{K}} = 0$. What value of $I_{\mathrm{app}}$ is needed to push the voltage from rest ($V_0 = -65$ mV) to $-40$ mV?
2. Now set $g_{\mathrm{K}} = 0.2$ µS. How much larger must $I_{\mathrm{app}}$ be to reach the same voltage? Why?
3. Can you find _any_ parameter combination that produces an action potential — where the voltage rises and falls back on its own, without sustained input? Why or why not?


In [ ]:
from widgets import fixed_conductance_widget

fixed_conductance_widget()

## Exercise 12.1b: Voltage-dependent gating introduces a threshold

Now we add a voltage-dependent activation gate $m$ to the sodium current, using the instantaneous approximation $m \approx m_\infty(V)$:

$$
C_{\mathrm{m}} \frac{\mathrm{d}V}{\mathrm{d}t} = -g_{\mathrm{Na}} \, m_\infty(V) \, (V - E_{\mathrm{Na}}) - g_{\mathrm{K}}(V - E_{\mathrm{K}}) + I_{\mathrm{app}}
$$

Use the widget below to explore this model. For each task, try to predict the answer analytically before using the widget to verify.

**Tasks:**

1. With the default parameters, what value of $I_{\mathrm{app}}$ is needed for the voltage to surpass 0 mV?
2. Set $I_{\mathrm{app}} = 0$. Increase $V_0$ slowly. At what value does the voltage suddenly jump to a high value instead of returning to rest? This is the **threshold**. Can you relate it to features of the $I(V)$ curve in the right panel?
3. Why can't this model generate a complete action potential (up _and_ back down)?


In [ ]:
from widgets import voltage_gated_conductance_widget

voltage_gated_conductance_widget()

## Exercise 12.1c: Implementing the full HH model

The widgets above showed that we need both activation _and_ inactivation/recovery mechanisms to generate a full action potential. The Hodgkin-Huxley model achieves this with three gating variables:

$$
C_{\mathrm{m}} \frac{\mathrm{d}V}{\mathrm{d}t} = - (I_{\mathrm{Na}} + I_{\mathrm{K}} + I_{\mathrm{L}}) + I_{\mathrm{app}}
$$

where:

$$I_{\mathrm{Na}} = \bar{g}_{\mathrm{Na}} \, m^3 \, h \, (V - E_{\mathrm{Na}})$$
$$I_{\mathrm{K}} = \bar{g}_{\mathrm{K}} \, n^4 \, (V - E_{\mathrm{K}})$$
$$I_{\mathrm{L}} = \bar{g}_{\mathrm{L}} \, (V - E_{\mathrm{L}})$$

and each gating variable follows:

$$\frac{\mathrm{d}m}{\mathrm{d}t} = \alpha_m(V)(1-m) - \beta_m(V) \, m$$
$$\frac{\mathrm{d}h}{\mathrm{d}t} = \alpha_h(V)(1-h) - \beta_h(V) \, h$$
$$\frac{\mathrm{d}n}{\mathrm{d}t} = \alpha_n(V)(1-n) - \beta_n(V) \, n$$

**Your task:** Complete the `rhs` function below by filling in the ionic currents and the derivatives.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


# Parameters (standard squid axon values)
Cm = 1.0  # µF/cm²
E_Na = 50.0  # mV
E_K = -77.0  # mV
E_L = -54.4  # mV
gNa = 120.0  # mS/cm²
gK = 36.0  # mS/cm²
gL = 0.3  # mS/cm²
I_amp = 10.0  # µA/cm² (stimulus amplitude)


def rhs(t, y):
    """Right-hand side of the HH equations."""
    V, m, h, n = y

    # Rate constants (these are provided for you)
    alpha_m = 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))
    beta_m = 4.0 * np.exp(-(V + 65.0) / 18.0)
    alpha_h = 0.07 * np.exp(-(V + 65.0) / 20.0)
    beta_h = 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))
    alpha_n = 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))
    beta_n = 0.125 * np.exp(-(V + 65.0) / 80.0)

    # Applied stimulus current (brief pulse from t=2 to t=3 ms)
    I_app = I_amp if (2 < t < 3) else 0.0

    # TODO: Calculate the ionic currents
    I_Na = ...  # hint: gNa * m^3 * h * (V - E_Na)
    I_K = ...
    I_L = ...

    # TODO: Calculate the derivatives
    dV_dt = ...  # hint: (- I_Na - I_K - I_L + I_app) / Cm
    dm_dt = ...
    dh_dt = ...
    dn_dt = ...

    return [dV_dt, dm_dt, dh_dt, dn_dt]

## Exercise 12.1d: Solving and plotting

Set the initial conditions (at rest, approximately $V = -65$ mV) and solve the system for 30 ms.

**Tasks:**

1. Plot the membrane potential $V(t)$. Does the shape match the "tug-of-war" description from the theory?
2. Plot all three gating variables ($m$, $h$, $n$) on the same axes. Can you identify which variable drives the upstroke, which drives repolarization, and which drives the afterhyperpolarization?
3. **Challenge:** Compute and plot the individual ionic currents ($I_{\mathrm{Na}}$, $I_{\mathrm{K}}$, $I_{\mathrm{L}}$) over time. Which current dominates during each phase of the AP?


In [ ]:
# Initial conditions: [V, m, h, n]
# At rest (-65 mV), m and n are near 0, h is near 0.6
V0 = -65.0
y0 = [V0, 0.05, 0.6, 0.32]

# Solve
t_span = (0, 30)
t_eval = np.linspace(*t_span, 2000)
sol = solve_ivp(rhs, t_span, y0, t_eval=t_eval)

# Unpack
V, m, h, n = sol.y
t = sol.t

In [ ]:
# 1. Plot the membrane potential
# Your code here

In [ ]:
# 2. Plot gating variables
# Your code here

In [ ]:
# 3. Plot ionic currents
# Your code here

## Exercise 12.1e: The refractory period

Modify the `I_app` logic in your `rhs` function to fire **two** identical stimulus pulses:

- First pulse: $t = 2$ to $t = 3$ ms
- Second pulse: $t = 5$ to $t = 6$ ms (i.e., 2 ms after the first pulse ends)

**Tasks:**

1. Run the simulation. Does the second stimulus trigger a second action potential?
2. Gradually increase the delay between the two pulses until the second stimulus _can_ trigger an AP. How long is the refractory period?
3. Plot the inactivation variable $h(t)$ alongside $V(t)$. Use this to explain _biologically_ why the cell cannot fire during the refractory period.


In [ ]:
# Your modified code here
# ...

## Exercise 12.1f: Interactive HH explorer

Use the widget below to explore how the HH model parameters affect the action potential. The default values are: $\bar{g}_{\mathrm{Na}} = 120$ mS/cm², $\bar{g}_{\mathrm{K}} = 36$ mS/cm², $\bar{g}_{\mathrm{L}} = 0.3$ mS/cm², $I_{\mathrm{app}} = 10$ µA/cm², stimulus at $t = 2$ ms for 1 ms, and $V_0 = -65$ mV.

**Tasks:**

1. Reset all parameters to their defaults. Decrease $I_{\mathrm{app}}$ until the stimulus no longer triggers an AP. What is the minimum current needed? (This is the **rheobase**.)
2. Reset to defaults. Decrease $\bar{g}_{\mathrm{Na}}$ while keeping all other parameters fixed. At what value does the cell fail to fire?
3. Reset to defaults. Increase $\bar{g}_{\mathrm{K}}$ while keeping all other parameters fixed. How does it affect the AP duration and the afterhyperpolarization?
4. Reset to defaults. Can you find a parameter combination that produces **repetitive firing** (multiple APs from a single sustained stimulus)? _Hint: try increasing the stimulus duration with moderate current._


In [ ]:
from widgets import hh_explorer_widget

hh_explorer_widget()